In [ ]:
import os, io, zipfile, requests, pandas as pd
from datetime import datetime
from google.cloud import storage

# Configurar o caminho da Service Account do GCP localmente
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../config/gcp_key.json"

pd.set_option('display.max_columns', None)  # Mostrar todas as colunas

# Definições de teste
ANO_MES     = "202401"
URL_CVM     = f"http://dados.cvm.gov.br/dados/FI/DOC/INF_DIARIO/DADOS/inf_diario_fi_{ANO_MES}.zip"
BUCKET_NAME = "cvm-landing-zone-bronze-dev" 
URL_CAD_FI  = "http://dados.cvm.gov.br/dados/FI/CAD/DADOS/cad_fi.csv"
URL_CDA     = f"http://dados.cvm.gov.br/dados/FI/DOC/CDA/DADOS/cda_fi_{ANO_MES}.zip"

In [ ]:
# --- CONFIGURAÇÕES DO FULL LOAD ---
ANO_ALVO = "2026"
MES_ATUAL = datetime.now().month  # Baixa até o mês corrente de 2026
BUCKET_NAME = "cvm-landing-zone-bronze-dev" # Confirme o nome do seu bucket
BASE_DIR = os.path.dirname(os.path.abspath(""))
KEY_PATH = os.path.join(BASE_DIR, "config", "gcp_key.json")

client = storage.Client.from_service_account_json(KEY_PATH)
bucket = client.bucket(BUCKET_NAME)

def upload_in_memory(url, gcs_path, content_type):
    """Auxiliar para streaming do arquivo em memória direto para o GCS"""
    response = requests.get(url, timeout=90)
    if response.status_code == 200:
        blob = bucket.blob(gcs_path)
        blob.upload_from_string(response.content, content_type=content_type)
        print(f" Success: gs://{BUCKET_NAME}/{gcs_path}")
        return True
    elif response.status_code == 404:
        print(f" Arquivo não encontrado na CVM (ainda não publicado): {url}")
        return False
    else:
        print(f" Erro HTTP {response.status_code} para {url}")
        return False

In [ ]:
# 1. CARGA: Cadastro de Fundos (cad_fi)
print("\n--- 1. Iniciando Full Load: Cad_FI ---")
hoje_str = datetime.now().strftime("%Y%m%d")
url_cad = "http://dados.cvm.gov.br/dados/FI/CAD/DADOS/cad_fi.csv"
path_cad = f"cvm/cad_fi/ano={ANO_ALVO}/mes={datetime.now().strftime('%m')}/cad_fi_{hoje_str}.csv"
upload_in_memory(url_cad, path_cad, "text/csv")

In [ ]:
# 2. CARGA: Informe Diário (inf_diario_fi) - 2026
print(f"\n--- 2. Iniciando Full Load: INF_DIARIO ({ANO_ALVO}) ---")
for mes in range(1, MES_ATUAL + 1):
    mes_str = f"{mes:02d}"
    ano_mes = f"{ANO_ALVO}{mes_str}"
    
    url_inf = f"http://dados.cvm.gov.br/dados/FI/DOC/INF_DIARIO/DADOS/inf_diario_fi_{ano_mes}.zip"
    path_inf = f"cvm/inf_diario_fi/ano={ANO_ALVO}/mes={mes_str}/inf_diario_fi_{ano_mes}.zip"
    upload_in_memory(url_inf, path_inf, "application/zip")

In [ ]:
# 3. CARGA: Carteira de Fundos (cda_fi) - 2026
print(f"\n--- 3. Iniciando Full Load: CDA_FI ({ANO_ALVO}) ---")
for mes in range(1, MES_ATUAL + 1):
    mes_str = f"{mes:02d}"
    ano_mes = f"{ANO_ALVO}{mes_str}"
    
    url_cda = f"http://dados.cvm.gov.br/dados/FI/DOC/CDA/DADOS/cda_fi_{ano_mes}.zip"
    path_cda = f"cvm/cda_fi/ano={ANO_ALVO}/mes={mes_str}/cda_fi_{ano_mes}.zip"
    upload_in_memory(url_cda, path_cda, "application/zip")

In [ ]:
# --- CONFIGURAÇÕES BCB / GCS ---
ANO_ALVO = "2026"
DATA_INICIAL = f"01/01/{ANO_ALVO}"
DATA_FINAL = datetime.now().strftime("%d/%m/%Y") # Vai até o dia de hoje

# Código 12 = CDI diário (% a.d.)
SERIE_CDI = 12
URL_BCB = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{SERIE_CDI}/dados?formato=json&dataInicial={DATA_INICIAL}&dataFinal={DATA_FINAL}"

BUCKET_NAME = "cvm-landing-zone-bronze-dev"
BASE_DIR = os.path.dirname(os.path.abspath(""))
KEY_PATH = os.path.join(BASE_DIR, "config", "gcp_key.json")

def ingest_bcb_cdi_bronze():
    client = storage.Client.from_service_account_json(KEY_PATH)
    bucket = client.bucket(BUCKET_NAME)
    
    print(f"Baixando dados do CDI (SGS {SERIE_CDI}) do BCB entre {DATA_INICIAL} e {DATA_FINAL}...")
    response = requests.get(URL_BCB, timeout=30)
    
    if response.status_code == 200:
        dados_json = response.content
        print(f"Download concluído! Total de registros recebidos: {len(response.json())}")
        
        # Caminho particionado no GCS
        gcs_path = f"bcb/cdi/ano={ANO_ALVO}/cdi_diario_{ANO_ALVO}.json"
        blob = bucket.blob(gcs_path)
        
        blob.upload_from_string(dados_json, content_type="application/json")
        print(f" Upload realizado com sucesso para: gs://{BUCKET_NAME}/{gcs_path}")
    else:
        print(f" Erro ao acessar API do BCB. Status code: {response.status_code}")
        
ingest_bcb_cdi_bronze()


In [ ]:
def testar_upload_gcs():
    client = storage.Client()
    bucket = client.bucket(BUCKET_NAME)
    
    # Definir o caminho particionado no estilo Hive
    ano = ANO_MES[:4]
    mes = ANO_MES[4:]
    gcs_path = f"cvm/inf_diario_fi/ano={ano}/mes={mes}/inf_diario_fi_{ANO_MES}.zip"
    
    blob = bucket.blob(gcs_path)
    
    # Enviar o conteúdo diretamente do download (sem precisar salvar em disco)
    blob.upload_from_string(response.content, content_type="application/zip")
    print(f" Upload realizado com sucesso para: gs://{BUCKET_NAME}/{gcs_path}")



In [ ]:
print(f"Baixando cadastro de fundos da CVM: {URL_CAD_FI}...")
response_cad = requests.get(URL_CAD_FI, timeout=30)

if response_cad.status_code == 200:
    print("Download do cad_fi.csv realizado com sucesso!")
    
    # Ler apenas as primeiras 10 linhas para preview
    df_cad_preview = pd.read_csv(
        io.BytesIO(response_cad.content),
        sep=";",
        encoding="ISO-8859-1"
    )
    
    # Mostrar as colunas essenciais que discutimos para o projeto
    colunas_cadastrais = [
        "CNPJ_FUNDO", "DENOM_SOCIAL", "SIT", "CLASS", 
        "GESTOR", "CNPJ_GESTOR", "ADMIN", "CNPJ_ADMIN", 
        "TAXA_ADM", "BENCHMARK"
    ]
    
    colunas_presentes = [c for c in colunas_cadastrais if c in df_cad_preview.columns]
else:
    print(f"Erro no download: Status {response_cad.status_code}")

In [ ]:
print(f"Baixando informe dos fundos da CVM: {URL_CVM}...")
response = requests.get(URL_CVM, timeout=30)

if response.status_code == 200:
    print("Download concluído com sucesso!")
    
    # Abrir o arquivo ZIP diretamente em memória
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        nome_arquivo_csv = z.namelist()[0]
        print(f"Arquivo encontrado dentro do ZIP: {nome_arquivo_csv}")
        
        # Ler apenas as primeiras 10 linhas para inspecionar o schema
        with z.open(nome_arquivo_csv) as f:
            df_preview = pd.read_csv(f, sep=";", encoding="ISO-8859-1")
            #testar_upload_gcs()
else:
    print(f"Falha no download. Status Code: {response.status_code}")
    

In [ ]:
print(f"Baixando carteira dos fundos (CDA_FI) para {ANO_MES}...")
res_cda = requests.get(URL_CDA, timeout=60)

if res_cda.status_code == 200:
    with zipfile.ZipFile(io.BytesIO(res_cda.content)) as z:
        print("Tabelas encontradas dentro do ZIP do CDA_FI:")
        for name in z.namelist():
            print(f" - {name}")
else:
    print(f"Erro no download: {res_cda.status_code}")

In [ ]:
with zipfile.ZipFile(io.BytesIO(res_cda.content)) as z:
    # 1. Carregar PL dos Fundos
    df_pl = pd.read_csv(z.open('cda_fi_PL_202401.csv'), sep=";", encoding="ISO-8859-1")
    df_pl = df_pl[['CNPJ_FUNDO_CLASSE', 'DENOM_SOCIAL', 'VL_PATRIM_LIQ']].drop_duplicates()
    df_pl = df_pl[df_pl['VL_PATRIM_LIQ'] > 0] # Remover fundos zerados

    # 2. BLC_2: Calcular exposição em FIDC (Cotas de Fundos)
    df_blc2 = pd.read_csv(z.open('cda_fi_BLC_2_202401.csv'), sep=";", encoding="ISO-8859-1")
    # Filtra por cotas onde o tipo de ativo indica FIDC
    fidc_pos = df_blc2[df_blc2['TP_ATIVO'].str.contains('FIDC', na=False, case=False)]
    fidc_sum = fidc_pos.groupby('CNPJ_FUNDO_CLASSE')['VL_MERC_POS_FINAL'].sum().reset_index()
    fidc_sum.rename(columns={'VL_MERC_POS_FINAL': 'VL_FIDC'}, inplace=True)

    # 3. BLC_4: Calcular exposição em Ações
    df_blc4 = pd.read_csv(z.open('cda_fi_BLC_4_202401.csv'), sep=";", encoding="ISO-8859-1")
    acoes_pos = df_blc4[df_blc4['TP_ATIVO'].str.contains('AÇÕES|ACOES|UNIT', na=False, case=False)]
    acoes_sum = acoes_pos.groupby('CNPJ_FUNDO_CLASSE')['VL_MERC_POS_FINAL'].sum().reset_index()
    acoes_sum.rename(columns={'VL_MERC_POS_FINAL': 'VL_ACOES'}, inplace=True)

    # 4. BLC_5 + BLC_6: Calcular exposição total em Crédito Privado
    df_blc5 = pd.read_csv(z.open('cda_fi_BLC_5_202401.csv'), sep=";", encoding="ISO-8859-1")
    df_blc6 = pd.read_csv(z.open('cda_fi_BLC_6_202401.csv'), sep=";", encoding="ISO-8859-1")
    
    credito_sum5 = df_blc5.groupby('CNPJ_FUNDO_CLASSE')['VL_MERC_POS_FINAL'].sum().reset_index()
    credito_sum6 = df_blc6.groupby('CNPJ_FUNDO_CLASSE')['VL_MERC_POS_FINAL'].sum().reset_index()
    
    credito_sum = pd.concat([credito_sum5, credito_sum6]).groupby('CNPJ_FUNDO_CLASSE')['VL_MERC_POS_FINAL'].sum().reset_index()
    credito_sum.rename(columns={'VL_MERC_POS_FINAL': 'VL_CREDITO'}, inplace=True)

# Unir todas as tabelas no DataFrame Principal
df_mestre = df_pl.merge(fidc_sum, on='CNPJ_FUNDO_CLASSE', how='left')
df_mestre = df_mestre.merge(acoes_sum, on='CNPJ_FUNDO_CLASSE', how='left')
df_mestre = df_mestre.merge(credito_sum, on='CNPJ_FUNDO_CLASSE', how='left').fillna(0)

# Calcular Percentuais em relação ao PL
df_mestre['PCT_CREDITO'] = df_mestre['VL_CREDITO'] / df_mestre['VL_PATRIM_LIQ']
df_mestre['PCT_FIDC'] = df_mestre['VL_FIDC'] / df_mestre['VL_PATRIM_LIQ']
df_mestre['PCT_ACOES'] = df_mestre['VL_ACOES'] / df_mestre['VL_PATRIM_LIQ']

# APLICAR OS REQUISITOS / FILTROS:
# 1. Pelo menos 10% em crédito
cond_credito = df_mestre['PCT_CREDITO'] >= 0.10

# 2. Sem 'INVESTIMENTO NO EXTERIOR' ou 'IE' no nome
cond_nome = ~df_mestre['DENOM_SOCIAL'].str.contains('INVESTIMENTO NO EXTERIOR| IE | EXTERIOR', case=False, regex=True, na=False)

# 3. Até 30% em FIDC
cond_fidc = df_mestre['PCT_FIDC'] <= 0.30

# 4. Não soberano (Garantir que não é 100% títulos públicos / tem mais de 10% de crédito)
# A própria condição de >= 10% crédito já desqualifica fundos 100% soberanos.

# 5. Ações até 3% do PL
cond_acoes = df_mestre['PCT_ACOES'] <= 0.03

# Filtrar a amostra final
df_fundos_credito_elegiveis = df_mestre[cond_credito & cond_nome & cond_fidc & cond_acoes]

print(f"Total de fundos analisados na amostra: {len(df_mestre)}")
print(f"Total de fundos de crédito elegíveis encontrados: {len(df_fundos_credito_elegiveis)}")

df_fundos_credito_elegiveis[['DENOM_SOCIAL', 'PCT_CREDITO', 'PCT_FIDC', 'PCT_ACOES']].head(10)

In [ ]:
df_fundos_credito_elegiveis